# MRMS × HydroFabric — Notebook 1 of 3: Read the HydroFabric and inventory everything

**Goal of the whole project**
1. Read the CONUS NextGen HydroFabric and extract every layer/feature.
2. Filter MRMS grid locations to those falling inside HydroFabric catchments.
3. Pick which dates/events to pull MRMS for.
4. Download 2-minute MRMS as CSV, each row tagged with: location, VPU, catchment (divide_id), and the nexus that catchment drains to.
5. (Later) filter locations by radar coverage.
6. (Later) filter VPUs to only those touched by the filtered flood events, then download MRMS for just those.

**This notebook = Step 1 only:** open `conus_nextgen.gpkg`, confirm I can read it,
inventory every layer and field, load the backbone layers, and build the master
catchment table (divide_id → VPU → nexus) that the rest of the project depends on.

This notebook is self-contained. At the end it saves everything downstream notebooks
need (`catchments_master`, `network`, `flowpaths`, `nexus`, `vpu_summary`) to disk, so
**Notebook 2** (MRMS↔HydroFabric crosswalk + event/AOI selection) can load them fresh
instead of relying on this kernel's memory.

## 1.0 Imports

The PROJ env fix below is the same defensive trick from the CUAHSI hydrofabric
notebook — it prevents a coordinate-database clash on JupyterHub. It must run
*before* geopandas/pyproj are imported, which is why it's the very first thing
in this cell.

In [1]:
import os
import sys
import warnings
from pathlib import Path

# --- PROJ database fix (must run BEFORE importing geopandas/pyproj) ---
proj_data = Path(sys.prefix) / "share" / "proj"
if proj_data.exists():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)

import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
import pyproj

pyproj.network.set_network_enabled(False)  # use offline transforms; avoids corrupt PROJ grid TIFFs

# pyogrio gives fast, memory-cheap layer info (no need to read the data).
# It ships with modern geopandas; we fall back to fiona if it's missing.
try:
    import pyogrio
    HAVE_PYOGRIO = True
except ImportError:
    HAVE_PYOGRIO = False

warnings.filterwarnings("ignore")

print("geopandas", gpd.__version__, "| pyogrio available:", HAVE_PYOGRIO)

geopandas 1.0.1 | pyogrio available: True


## 1.1 Locate the GeoPackage and confirm it exists

Set the path once here. The CONUS file is large, so we check it's present and
report its size before touching it.

In [2]:
# Adjust if your file lives elsewhere. We check a couple of common spots.
CANDIDATES = [
    Path("conus_nextgen.gpkg"),
    Path("storm_usgs_data/conus_nextgen.gpkg"),
]

gpkg_path = next((p for p in CANDIDATES if p.exists()), None)

if gpkg_path is None:
    raise FileNotFoundError(
        "conus_nextgen.gpkg not found. Edit CANDIDATES with the correct path.\n"
        f"Looked in: {[str(p) for p in CANDIDATES]}"
    )

size_gb = gpkg_path.stat().st_size / 1e9
print(f"Found: {gpkg_path.resolve()}")
print(f"Size : {size_gb:.2f} GB")

Found: /home/jovyan/Group_Project/conus_nextgen.gpkg
Size : 4.90 GB


## 1.2 List every layer, then inventory each one

"Extract every feature" = see every layer, and for each layer its fields,
geometry type, CRS, and row count. We use `pyogrio.read_info`, which reads only
metadata (fast, no data loaded into memory).

In [3]:
layers = fiona.listlayers(str(gpkg_path))
print(f"{len(layers)} layers found:\n")
for name in layers:
    print("  -", name)

10 layers found:

  - flowpaths
  - divides
  - lakes
  - hydrolocations
  - nexus
  - pois
  - flowpath-attributes
  - flowpath-attributes-ml
  - network
  - divide-attributes


In [4]:
def layer_info(path, layer):
    """Return (n_features, geometry_type, crs, [field names]) cheaply."""
    if HAVE_PYOGRIO:
        info = pyogrio.read_info(str(path), layer=layer)
        crs = info.get("crs")
        return (
            int(info.get("features", -1)),
            info.get("geometry_type"),
            str(crs) if crs else None,
            list(info.get("fields", [])),
        )
    # Fallback via fiona
    with fiona.open(str(path), layer=layer) as src:
        return (len(src), src.schema.get("geometry"),
                str(src.crs), list(src.schema["properties"].keys()))

rows = []
fields_by_layer = {}
for name in layers:
    n, geom, crs, fields = layer_info(gpkg_path, name)
    fields_by_layer[name] = fields
    rows.append({"layer": name, "n_features": n,
                 "geometry": geom, "crs": crs, "n_fields": len(fields)})

inventory = pd.DataFrame(rows)
inventory

,layer,n_features,geometry,crs,n_fields
0,flowpaths,828288,Unknown,EPSG:5070,12
1,divides,831777,Polygon,EPSG:5070,10
2,lakes,5408,Point,EPSG:5070,23
3,hydrolocations,151469,Point,EPSG:5070,11
4,nexus,409122,Point,EPSG:5070,5
5,pois,104830,None,None,4
6,flowpath-attributes,828689,None,None,21
7,flowpath-attributes-ml,828689,None,None,23
8,network,3461367,None,None,19
9,divide-attributes,830357,None,None,40


In [5]:
# The fields (columns) available in each layer — this is the "every feature" map.
for name in layers:
    print(f"\n=== {name} ({len(fields_by_layer[name])} fields) ===")
    print(", ".join(fields_by_layer[name]))


=== flowpaths (12 fields) ===
id, toid, mainstem, order, hydroseq, lengthkm, areasqkm, tot_drainage_areasqkm, has_divide, divide_id, poi_id, vpuid

=== divides (10 fields) ===
divide_id, toid, type, ds_id, areasqkm, vpuid, id, lengthkm, tot_drainage_areasqkm, has_flowline

=== lakes (23 fields) ===
lake_id, LkArea, LkMxE, WeirC, WeirL, OrificeC, OrificeA, OrificeE, WeirE, ifd, Dam_Length, domain, poi_id, hf_id, reservoir_index_AnA, reservoir_index_Extended_AnA, reservoir_index_GDL_AK, reservoir_index_Medium_Range, reservoir_index_Short_Range, res_id, vpuid, lake_x, lake_y

=== hydrolocations (11 fields) ===
poi_id, id, nex_id, hf_id, hl_link, hl_reference, hl_uri, hl_source, hl_x, hl_y, vpuid

=== nexus (5 fields) ===
id, toid, type, vpuid, poi_id

=== pois (4 fields) ===
poi_id, id, nex_id, vpuid

=== flowpath-attributes (21 fields) ===
link, to, Length_m, Y, n, nCC, BtmWdth, TopWdth, TopWdthCC, ChSlp, alt, So, MusX, MusK, gage, gage_nex_id, WaterbodyID, waterbody_nex_id, id, toid, v

## 1.3 Load the backbone layers

We need:
- `network`  — attribute table linking divide_id ↔ flowpath id ↔ toid ↔ vpuid (no geometry, cheap)
- `divides`  — catchment polygons (this is what MRMS points get joined to)
- `flowpaths`— stream reaches (kept for later / radar-coverage and routing context)
- `nexus`    — junction points where catchments discharge

**Memory note:** `divides`/`flowpaths` carry geometry for all of CONUS, so we read
only the columns we need. Loading takes a bit. Comment out `flowpaths` if RAM is tight.

In [6]:
# Attribute-only table: fast, full read.
network = gpd.read_file(
    gpkg_path, layer="network",
    columns=["id", "toid", "divide_id", "vpuid"],
    read_geometry=False,
)

# Polygons / lines — restrict columns to keep memory down.
divides = gpd.read_file(
    gpkg_path, layer="divides",
    columns=["divide_id", "id", "toid", "areasqkm"],
)

flowpaths = gpd.read_file(
    gpkg_path, layer="flowpaths",
    columns=["id", "toid"],
)

nexus = gpd.read_file(
    gpkg_path, layer="nexus",
    columns=["id", "toid"],
)

print(f"network  : {len(network):,} rows (no geometry)")
print(f"divides  : {len(divides):,} polygons  | CRS {divides.crs}")
print(f"flowpaths: {len(flowpaths):,} lines    | CRS {flowpaths.crs}")
print(f"nexus    : {len(nexus):,} points   | CRS {nexus.crs}")

network  : 3,461,367 rows (no geometry)
divides  : 831,777 polygons  | CRS EPSG:5070
flowpaths: 828,288 lines    | CRS EPSG:5070
nexus    : 409,122 points   | CRS EPSG:5070


In [7]:
# Quick peek so we can see real values/ID formats (cat-..., nex-..., wb-...).
display(network.head(3))
display(divides.head(3))
display(nexus.head(3))

,id,toid,divide_id,vpuid
0,wb-20469,tnx-1000000125,cat-20469,01
1,wb-20469,tnx-1000000125,cat-20469,01
2,wb-20469,tnx-1000000125,cat-20469,01


,divide_id,toid,areasqkm,id,geometry
0,cat-276,cnx-276,27.454477,None,"POLYGON ((2215875 2737364.998, 2215754.998 273..."
1,cat-275,cnx-275,25.207642,None,"POLYGON ((2229944.998 2745794.998, 2229825.002..."
2,cat-274,cnx-274,16.707601,None,"POLYGON ((2209485.004 2721074.997, 2209484.997..."


,id,toid,geometry
0,nex-1000,wb-1000,POINT (2065964.551 2865305.933)
1,nex-10000,wb-10000,POINT (2028886.179 2457811.216)
2,nex-10004,wb-10004,POINT (2031522.75 2455486.067)


## 1.4 Build the master catchment table (divide_id → VPU → nexus)

This is the key product of Step 1. Each catchment polygon gets:
- `divide_id`  — the catchment number
- `vpuid`      — which VPU it belongs to (drives event-based VPU filtering later)
- `nexus_id`   — the nexus it drains into (a catchment's `toid` is its `nex-...`)
- `areasqkm`, `geometry`

Later steps spatially join MRMS grid points **into** this table, so every MRMS
location inherits its catchment, VPU, and nexus in one shot. A `radar_covered`
flag will be added here in a later step (placeholder noted, not built yet).

In [8]:
# network can have several rows per divide_id; keep one vpuid per divide.
net_lookup = (
    network.dropna(subset=["divide_id"])
           .drop_duplicates(subset=["divide_id"])[["divide_id", "vpuid"]]
)

catchments_master = (
    divides
    .rename(columns={"id": "flowpath_id", "toid": "nexus_id"})
    .merge(net_lookup, on="divide_id", how="left")
)

# Sanity checks
n = len(catchments_master)
print(f"catchments_master: {n:,} catchments")
print(f"  with a VPU     : {catchments_master['vpuid'].notna().sum():,}")
print(f"  with a nexus   : {catchments_master['nexus_id'].astype(str).str.startswith('nex-').sum():,}")
print(f"  unique VPUs    : {catchments_master['vpuid'].nunique()}")

catchments_master.head()

catchments_master["nexus_type"] = (
    catchments_master["nexus_id"]
    .astype(str)
    .str.extract(r"^([a-z]+)-")[0]
)

catchments_master["is_terminal"] = catchments_master["nexus_type"].isin(
    ["tnx", "cnx", "inx"]
)

catchments_master: 831,777 catchments
  with a VPU     : 830,357
  with a nexus   : 811,570
  unique VPUs    : 21


## 1.5 What VPUs exist (sets up event-based filtering later)

Later you'll intersect your flood events with these VPUs and keep only the ones
that contain events. For now, just confirm what's available and how big each is.

In [9]:
vpu_summary = (
    catchments_master.groupby("vpuid")
    .agg(n_catchments=("divide_id", "size"),
         area_km2=("areasqkm", "sum"))
    .sort_index()
)
print(f"{len(vpu_summary)} VPUs")
vpu_summary

21 VPUs


,n_catchments,area_km2
vpuid,,
01,20567,169506.945673
02,35494,277762.650208
03N,31326,251265.129942
03S,13911,178577.782077
03W,30675,240450.490067
04,35894,317467.391260
05,51581,421958.454825
06,14167,105953.986092
07,57595,492039.507872


## Step 1b — Optional terminal outlet nexus

This step adds `terminal_nexus_id`, which is the final basin outlet reached by walking `toid` downstream until water exits the hydrofabric domain at `wb-0`.

The immediate `nexus_id` is the main column needed for Steps 2–4. The terminal outlet is optional and mainly useful for later routing analysis.

A few catchments may sit in residual network cycles. These are flagged with `terminal_is_clean = False`.

In [10]:
nexus_next = dict(
    zip(
        nexus["id"].astype(str),
        nexus["toid"].astype(str)
    )
)

net_wb = network[network["id"].astype(str).str.startswith("wb-")]

wb_next = dict(
    zip(
        net_wb["id"].astype(str),
        net_wb["toid"].astype(str)
    )
)

TERMINAL_WB = "wb-0"


def _down_nexus(n):
    wb = nexus_next.get(n)

    if wb is None or wb == TERMINAL_WB:
        return None

    return wb_next.get(wb)


terminal_cache = {}


def terminal_of(start):
    path = []
    cur = start
    seen = set()

    while cur is not None and cur not in terminal_cache:

        if cur in seen:
            # cycle guard
            for p in path:
                terminal_cache[p] = cur
            return cur

        seen.add(cur)
        path.append(cur)

        nxt = _down_nexus(cur)

        if nxt is None:
            terminal_cache[cur] = cur
            break

        cur = nxt

    result = terminal_cache.get(cur, cur)

    for p in path:
        terminal_cache[p] = result

    return result


for n in set(catchments_master["nexus_id"].astype(str)):
    terminal_of(n)


catchments_master["terminal_nexus_id"] = (
    catchments_master["nexus_id"]
    .astype(str)
    .map(terminal_cache)
)

catchments_master["terminal_is_clean"] = ~(
    catchments_master["terminal_nexus_id"]
    .astype(str)
    .str.startswith("nex-")
)

print(
    "terminal outlets:",
    catchments_master["terminal_nexus_id"].nunique(),
    "| cycle-flagged:",
    (~catchments_master["terminal_is_clean"]).sum()
)

terminal outlets: 17726 | cycle-flagged: 5


## 1.6 Save outputs for Notebook 2

Everything downstream (the MRMS↔HydroFabric crosswalk, HUC8/event selection, and
the per-event MRMS pipeline) is built on top of `catchments_master`, `network`,
`flowpaths`, and `nexus`. Save them here so Notebook 2 can load them without
re-reading the full CONUS GeoPackage.

- `catchments_master` and `flowpaths`/`nexus` carry geometry → saved as GeoParquet.
- `network` has no geometry → saved as plain parquet.
- `vpu_summary` is saved as CSV for a quick look outside Python.

In [11]:
OUT_DIR = Path("hydrofabric_outputs")
OUT_DIR.mkdir(exist_ok=True)

catchments_master.to_parquet(OUT_DIR / "catchments_master.parquet")
network.to_parquet(OUT_DIR / "network.parquet")
flowpaths.to_parquet(OUT_DIR / "flowpaths.parquet")
nexus.to_parquet(OUT_DIR / "nexus.parquet")
vpu_summary.to_csv(OUT_DIR / "vpu_summary.csv")

print("Saved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.iterdir()):
    print(" -", f.name, f"({f.stat().st_size / 1e6:.1f} MB)")

Saved to /home/jovyan/Group_Project/hydrofabric_outputs
 - catchments_master.parquet (698.1 MB)
 - flowpaths.parquet (932.3 MB)
 - network.parquet (22.9 MB)
 - nexus.parquet (11.2 MB)
 - vpu_summary.csv (0.0 MB)


## Notebook 1 — done ✅

I can read `conus_nextgen.gpkg`, I've inventoried every layer and field, and I
built `catchments_master` (divide_id → VPU → nexus + geometry + terminal outlet),
which is the table everything downstream attaches to. All of it is saved to
`hydrofabric_outputs/` for the next notebook.

**Next: Notebook 2** — build the MRMS 1 km grid points, spatially join them into
`catchments_master` so each MRMS cell gets a catchment/VPU/nexus, build the
correctness visuals, then select a HUC8/event subset to carry into Notebook 3.